# El Geometrisi ile Kimlik Doğrulama

**Biyometri:** El geometrisi  
**Veri Seti:** 11k Hands Dataset (Kaggle)  
**Yöntem:** MediaPipe ile iskelet çıkarımı + SVM / MLP sınıflandırma  
**Değerlendirme:** Accuracy, FAR, FRR, EER, Confusion Matrix

---
**Çalıştırma sırası:**
1. Runtime → Change runtime type → **T4 GPU**
2. Hücreleri sırayla çalıştır
3. Özellik çıkarımı (~10 dk) bir kez yapılır, Drive'a kaydedilir — kesilirse kaldığı yerden devam eder

In [ ]:
# ── Hücre 1: Kurulum ──────────────────────────────────────────────────────────
!pip install mediapipe kagglehub -q

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/odev_output'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive bağlandı:', DRIVE_DIR)

In [ ]:
# ── Hücre 2: Dataset + MediaPipe Modeli İndir ─────────────────────────────────
import os, shutil, urllib.request

# ↓↓↓ Kaggle token'ını buraya yapıştır ↓↓↓
KAGGLE_TOKEN = 'KGAT_buraya_kendi_tokenini_yapistir'
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

import kagglehub
DATA_DIR   = '/content/data/11k_hands'
IMAGE_ROOT = os.path.join(DATA_DIR, 'Hands', 'Hands')
os.makedirs(IMAGE_ROOT, exist_ok=True)

print('Dataset indiriliyor...')
cache = kagglehub.dataset_download('shyambhu/hands-and-palm-images-dataset')
copied = 0
for root, _, fnames in os.walk(cache):
    for fname in fnames:
        src = os.path.join(root, fname)
        dst = os.path.join(
            IMAGE_ROOT if fname.lower().endswith(('.jpg','.jpeg','.png')) else DATA_DIR,
            fname
        )
        if not os.path.exists(dst):
            shutil.copy2(src, dst); copied += 1
print(f'  {copied} dosya kopyalandı | Görüntü: {len(os.listdir(IMAGE_ROOT))}')

# MediaPipe hand_landmarker.task modelini indir
MODEL_PATH = '/content/hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('MediaPipe modeli indiriliyor...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        MODEL_PATH
    )
print(f'  hand_landmarker.task hazır ({os.path.getsize(MODEL_PATH)//1024} KB)')

In [ ]:
# ── Hücre 3: Import'lar ve Sabitler ──────────────────────────────────────────
import math, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from scipy.spatial.distance import euclidean

HANDINFO_CSV = os.path.join(DATA_DIR, 'HandInfo.csv')
FEATURES_CSV = os.path.join(DRIVE_DIR, 'features.csv')   # Drive'a kaydedilir
PLOTS_DIR    = os.path.join(DRIVE_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

MIN_SAMPLES  = 10
TEST_SIZE    = 0.30
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Hazır.')

In [ ]:
# ── Hücre 4: Özellik Tanımları ────────────────────────────────────────────────
WRIST=0; THUMB=[1,2,3,4]; INDEX=[5,6,7,8]; MIDDLE=[9,10,11,12]
RING=[13,14,15,16]; PINKY=[17,18,19,20]
FINGERS=[THUMB,INDEX,MIDDLE,RING,PINKY]
FINGERTIPS=[4,8,12,16,20]

FEATURE_NAMES = (
    ['len_thumb','len_index','len_middle','len_ring','len_pinky']
    + [f'tip_dist_{a}_{b}' for i,a in enumerate(FINGERTIPS) for b in FINGERTIPS[i+1:]]
    + ['palm_w_idx_mid','palm_w_mid_rng','palm_w_rng_pnk','palm_w_full']
    + ['ratio_thumb','ratio_index','ratio_middle','ratio_ring','ratio_pinky']
    + ['angle_pip_index','angle_pip_middle','angle_pip_ring','angle_pip_pinky']
    + ['wrist_dist_thumb','wrist_dist_index','wrist_dist_middle','wrist_dist_ring','wrist_dist_pinky']
    + [f'lm_{i}{"x" if j==0 else "y"}' for i in range(21) for j in range(2)]
)

def _dist(a, b): return math.dist(a, b)

def _angle(p1, vertex, p2):
    v1 = (p1[0]-vertex[0], p1[1]-vertex[1])
    v2 = (p2[0]-vertex[0], p2[1]-vertex[1])
    dot = v1[0]*v2[0]+v1[1]*v2[1]
    m1 = math.sqrt(v1[0]**2+v1[1]**2); m2 = math.sqrt(v2[0]**2+v2[1]**2)
    return math.acos(max(-1.0, min(1.0, dot/(m1*m2)))) if m1>1e-9 and m2>1e-9 else 0.0

def normalize_landmarks(lm_array):
    pts = lm_array.copy()
    pts -= pts[WRIST]
    ref = _dist(pts[WRIST], pts[MIDDLE[0]])
    return pts/ref if ref > 1e-9 else None

def extract_features(pts):
    feats = []
    fl = [sum(_dist(pts[f[i]],pts[f[i+1]]) for i in range(len(f)-1)) for f in FINGERS]
    feats.extend(fl)
    for i,a in enumerate(FINGERTIPS):
        for b in FINGERTIPS[i+1:]: feats.append(_dist(pts[a],pts[b]))
    palm = _dist(pts[5],pts[17])
    feats += [_dist(pts[5],pts[9]),_dist(pts[9],pts[13]),_dist(pts[13],pts[17]),palm]
    feats += [f/palm if palm>1e-9 else 0.0 for f in fl]
    for s,v,e in [(5,6,7),(9,10,11),(13,14,15),(17,18,19)]: feats.append(_angle(pts[s],pts[v],pts[e]))
    for t in FINGERTIPS: feats.append(_dist(pts[WRIST],pts[t]))
    for i in range(21): feats.extend([pts[i][0],pts[i][1]])
    return feats

print(f'Özellik sayısı: {len(FEATURE_NAMES)}')

In [ ]:
# ── Hücre 5: MediaPipe Özellik Çıkarımı ──────────────────────────────────────
# Eğer features.csv zaten Drive'da varsa bu hücreyi atla

if os.path.exists(FEATURES_CSV):
    print(f'features.csv zaten mevcut ({FEATURES_CSV}), çıkarım atlanıyor.')
else:
    info    = pd.read_csv(HANDINFO_CSV)
    img_col = next(c for c in info.columns if 'image' in c.lower() or 'file' in c.lower())
    id_col  = next(c for c in info.columns if 'id' in c.lower() and 'image' not in c.lower())
    print(f'HandInfo: {len(info)} görüntü | img_col={img_col} | id_col={id_col}')

    with open(MODEL_PATH,'rb') as f: model_bytes = f.read()
    opts = mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_buffer=model_bytes),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    detector = mp_vision.HandLandmarker.create_from_options(opts)

    rows, failed = [], []
    for _, row in tqdm(info.iterrows(), total=len(info), desc='Özellik çıkarımı'):
        img_name = str(row[img_col]).strip()
        subj_id  = str(row[id_col]).strip()
        img_path = os.path.join(IMAGE_ROOT, img_name)
        if not os.path.exists(img_path): failed.append(img_name); continue

        with open(img_path,'rb') as f:
            img_bgr = cv2.imdecode(np.frombuffer(f.read(),np.uint8), cv2.IMREAD_COLOR)
        if img_bgr is None: failed.append(img_name); continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_img  = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        result  = detector.detect(mp_img)

        if not result.hand_landmarks:
            gray   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                              data=cv2.cvtColor(cv2.equalizeHist(gray), cv2.COLOR_GRAY2RGB))
            result = detector.detect(mp_img)

        if not result.hand_landmarks: failed.append(img_name); continue

        h,w = img_bgr.shape[:2]
        pts = np.array([[l.x*w, l.y*h] for l in result.hand_landmarks[0]], dtype=float)
        norm_pts = normalize_landmarks(pts)
        if norm_pts is None: failed.append(img_name); continue
        rows.append([img_name, subj_id] + extract_features(norm_pts))

    detector.close()
    pd.DataFrame(rows, columns=['image_name','subject_id']+FEATURE_NAMES).to_csv(FEATURES_CSV, index=False)
    print(f'\nTamamlandı: {len(rows)} başarılı | {len(failed)} başarısız')
    print(f'Kaydedildi: {FEATURES_CSV}')

In [ ]:
# ── Hücre 6: Veri Yükleme Fonksiyonu ─────────────────────────────────────────
def load_data(aspect_filter=None):
    df = pd.read_csv(FEATURES_CSV)
    if aspect_filter:
        info = pd.read_csv(HANDINFO_CSV)
        img_col = next(c for c in info.columns if 'image' in c.lower() or 'file' in c.lower())
        asp_col = next(c for c in info.columns if 'aspect' in c.lower())
        info = info.rename(columns={img_col:'image_name'})
        df   = df.merge(info[['image_name', asp_col]], on='image_name', how='left')
        df   = df[df[asp_col].str.contains(aspect_filter, case=False, na=False)]
        df   = df.drop(columns=[asp_col])

    counts = df['subject_id'].value_counts()
    df     = df[df['subject_id'].isin(counts[counts >= MIN_SAMPLES].index)]
    label  = aspect_filter if aspect_filter else 'tum'

    feat_cols = [c for c in df.columns if c not in ('image_name','subject_id')]
    X = df[feat_cols].values.astype(float)
    y = LabelEncoder().fit_transform(df['subject_id'].values)
    print(f'[{label}] {len(df)} örnek | {len(np.unique(y))} kişi | {X.shape[1]} özellik')
    return X, y, feat_cols

def split_scale(X, y):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=TEST_SIZE,
                                        stratify=y, random_state=RANDOM_STATE)
    sc  = StandardScaler()
    return sc.fit_transform(Xtr), sc.transform(Xte), ytr, yte

# Veriyi yükle
X_all,    y_all,    _    = load_data(None)
X_dorsal, y_dorsal, _    = load_data('dorsal')
X_palmar, y_palmar, feat = load_data('palmar')

In [ ]:
# ── Hücre 7: FAR / FRR / EER ──────────────────────────────────────────────────
def compute_far_frr(Xtr, ytr, Xte, yte, n=500):
    classes   = np.unique(ytr)
    centroids = {c: Xtr[ytr==c].mean(0) for c in classes}
    gen, imp  = [], []
    rng = np.random.default_rng(RANDOM_STATE)
    for i in range(len(Xte)):
        lbl = yte[i]
        if lbl not in centroids: continue
        gen.append(-euclidean(Xte[i], centroids[lbl]))
        others = [c for c in classes if c != lbl]
        for c in rng.choice(others, size=min(10,len(others)), replace=False):
            imp.append(-euclidean(Xte[i], centroids[c]))
    gen, imp = np.array(gen), np.array(imp)
    thr  = np.linspace(np.concatenate([gen,imp]).min(), np.concatenate([gen,imp]).max(), n)
    far  = np.array([np.mean(imp>=t) for t in thr])
    frr  = np.array([np.mean(gen< t) for t in thr])
    idx  = np.argmin(np.abs(far-frr))
    eer  = (far[idx]+frr[idx])/2
    return thr, far, frr, eer, thr[idx], gen, imp

print('FAR/FRR fonksiyonu hazır.')

In [ ]:
# ── Hücre 8: Grafik Fonksiyonları ─────────────────────────────────────────────
def plot_confusion_matrix(yte, ypred, tag, top_n=20):
    top = pd.Series(yte).value_counts().head(top_n).index.tolist()
    mask = np.isin(yte, top)
    cm   = confusion_matrix(yte[mask], ypred[mask], labels=top)
    fig, ax = plt.subplots(figsize=(13,11))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=top, yticklabels=top, ax=ax, linewidths=0.3)
    ax.set_xlabel('Tahmin Edilen'); ax.set_ylabel('Gerçek')
    ax.set_title(f'Confusion Matrix — {tag} (İlk {top_n} Sınıf)')
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f'cm_{tag}.png')
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f'  [grafik] {os.path.basename(path)}')

def plot_far_frr(thr, far, frr, eer, eer_thr, tag):
    fig, ax = plt.subplots(figsize=(9,6))
    ax.plot(thr, far, color='red',  lw=2, label='FAR (Yanlış Kabul)')
    ax.plot(thr, frr, color='blue', lw=2, label='FRR (Yanlış Red)')
    ax.axvline(eer_thr, color='green', ls='--', lw=1.5, label=f'EER = {eer*100:.2f}%')
    ax.set_xlabel('Eşik Değeri'); ax.set_ylabel('Hata Oranı')
    ax.set_title(f'FAR & FRR — {tag}'); ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f'far_frr_{tag}.png')
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f'  [grafik] {os.path.basename(path)}')

def plot_score_dist(gen, imp, eer_thr, tag):
    fig, ax = plt.subplots(figsize=(9,6))
    ax.hist(imp, bins=60, alpha=0.55, color='red',  label='Sahte (Impostor)', density=True)
    ax.hist(gen, bins=60, alpha=0.55, color='blue', label='Gerçek (Genuine)',  density=True)
    ax.axvline(eer_thr, color='green', ls='--', lw=1.5, label=f'EER Eşiği = {eer_thr:.3f}')
    ax.set_xlabel('Skor'); ax.set_ylabel('Yoğunluk')
    ax.set_title(f'Genuine vs Impostor Dağılımı — {tag}'); ax.legend(); ax.grid(alpha=0.4)
    plt.tight_layout()
    path = os.path.join(PLOTS_DIR, f'score_dist_{tag}.png')
    plt.savefig(path, dpi=150); plt.show(); plt.close()
    print(f'  [grafik] {os.path.basename(path)}')

print('Grafik fonksiyonları hazır.')

In [ ]:
# ── Hücre 9: Model Eğitimi ────────────────────────────────────────────────────
def run_experiment(tag, X, y):
    print(f'\n{"+"*55}')
    print(f'  {tag}')
    print(f'{"+"*55}')
    Xtr, Xte, ytr, yte = split_scale(X, y)

    results = {}

    # ── SVM (GridSearchCV) ──
    print('  SVM GridSearchCV (C x gamma, cv=3)...')
    gs = GridSearchCV(
        SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, decision_function_shape='ovr'),
        {'C':[1,10,100], 'gamma':['scale',0.01,0.001]},
        cv=3, scoring='accuracy', n_jobs=-1
    )
    gs.fit(Xtr, ytr)
    svm_pred = gs.best_estimator_.predict(Xte)
    svm_acc  = accuracy_score(yte, svm_pred)
    svm_f1   = f1_score(yte, svm_pred, average='macro', zero_division=0)
    thr,far,frr,svm_eer,eer_thr,gen,imp = compute_far_frr(Xtr,ytr,Xte,yte)
    print(f'  SVM → Acc: {svm_acc*100:.2f}%  F1: {svm_f1*100:.2f}%  EER: {svm_eer*100:.2f}%  Best: {gs.best_params_}')
    plot_confusion_matrix(yte, svm_pred, f'SVM_{tag}')
    plot_far_frr(thr, far, frr, svm_eer, eer_thr, f'SVM_{tag}')
    plot_score_dist(gen, imp, eer_thr, f'SVM_{tag}')
    results['SVM'] = {'Accuracy':f'{svm_acc*100:.2f}%','Macro F1':f'{svm_f1*100:.2f}%',
                      'EER':f'{svm_eer*100:.2f}%','Params':str(gs.best_params_)}

    # ── MLP ──
    print('  MLP (256-128-64, adam)...')
    mlp = MLPClassifier(
        hidden_layer_sizes=(256,128,64), activation='relu', solver='adam',
        alpha=0.001, batch_size=64, learning_rate='adaptive', learning_rate_init=0.001,
        max_iter=300, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=20, random_state=RANDOM_STATE, verbose=False
    )
    mlp.fit(Xtr, ytr)
    mlp_pred = mlp.predict(Xte)
    mlp_acc  = accuracy_score(yte, mlp_pred)
    mlp_f1   = f1_score(yte, mlp_pred, average='macro', zero_division=0)
    thr,far,frr,mlp_eer,eer_thr,gen,imp = compute_far_frr(Xtr,ytr,Xte,yte)
    print(f'  MLP → Acc: {mlp_acc*100:.2f}%  F1: {mlp_f1*100:.2f}%  EER: {mlp_eer*100:.2f}%  ({mlp.n_iter_} iter)')
    plot_confusion_matrix(yte, mlp_pred, f'MLP_{tag}')
    plot_far_frr(thr, far, frr, mlp_eer, eer_thr, f'MLP_{tag}')
    plot_score_dist(gen, imp, eer_thr, f'MLP_{tag}')
    results['MLP'] = {'Accuracy':f'{mlp_acc*100:.2f}%','Macro F1':f'{mlp_f1*100:.2f}%',
                      'EER':f'{mlp_eer*100:.2f}%','Params':'256-128-64, adam'}

    return results

print('Eğitim fonksiyonu hazır.')

In [ ]:
# ── Hücre 10: Tüm Deneyleri Çalıştır ─────────────────────────────────────────
all_results = {}
all_results['tum']    = run_experiment('tum',    X_all,    y_all)
all_results['dorsal'] = run_experiment('dorsal', X_dorsal, y_dorsal)
all_results['palmar'] = run_experiment('palmar', X_palmar, y_palmar)

In [ ]:
# ── Hücre 11: Sonuç Tablosu + Karşılaştırma Grafikleri ───────────────────────
rows = []
for aspect, models in all_results.items():
    for mname, metrics in models.items():
        rows.append({'Deney':f'{mname}_{aspect}', **metrics})
df_res = pd.DataFrame(rows)

# CSV olarak Drive'a kaydet
csv_path = os.path.join(DRIVE_DIR, 'results_summary.csv')
df_res.to_csv(csv_path, index=False)

print('\n' + '='*55)
print('ÖZET SONUÇ TABLOSU')
print('='*55)
display(df_res)

# ── Karşılaştırma Bar Grafikleri ──
aspects  = ['tum','dorsal','palmar']
labels   = ['Tüm Veri','Dorsal','Palmar']
models   = ['SVM','MLP']
colors   = ['#2196F3','#FF5722']
x        = np.arange(len(aspects))
width    = 0.3

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,6))
for i,(m,c) in enumerate(zip(models,colors)):
    accs = [float(all_results[a][m]['Accuracy'].strip('%')) for a in aspects]
    bars = ax1.bar(x+(i-0.5)*width, accs, width, label=m, color=c, edgecolor='white')
    [ax1.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}', ha='center', fontsize=9)
     for b,v in zip(bars,accs)]
ax1.set_xticks(x); ax1.set_xticklabels(labels); ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Model Karşılaştırması — Accuracy'); ax1.legend(); ax1.set_ylim(60,100); ax1.grid(axis='y',alpha=0.4)

for i,(m,c) in enumerate(zip(models,colors)):
    eers = [float(all_results[a][m]['EER'].strip('%')) for a in aspects]
    bars = ax2.bar(x+(i-0.5)*width, eers, width, label=m, color=c, edgecolor='white')
    [ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}', ha='center', fontsize=9)
     for b,v in zip(bars,eers)]
ax2.set_xticks(x); ax2.set_xticklabels(labels); ax2.set_ylabel('EER (%) — düşük = iyi')
ax2.set_title('Model Karşılaştırması — EER'); ax2.legend(); ax2.grid(axis='y',alpha=0.4)

plt.tight_layout()
path = os.path.join(PLOTS_DIR,'comparison.png')
plt.savefig(path, dpi=150); plt.show(); plt.close()
print(f'\nTüm sonuçlar kaydedildi: {DRIVE_DIR}')